# HIT THE VIBE — Kiểm tra NFR01–NFR05 trên Kaggle CPU
Chọn **Python**, **Accelerator: None**, **Internet: On** và chạy các ô từ trên xuống.
Notebook chứa sẵn mã nguồn, không cần clone/push GitHub. Mỗi lượt tạo database mới;
không sử dụng hoặc thay đổi `concert.db` của bạn. Mã backend được đóng gói từ dự án hiện tại.

| Yêu cầu | Cách kiểm tra | Tiêu chí đánh giá |
|---|---|---|
| NFR01 | Chromium đo tải mới HTML/CSS/JS, API phụ thuộc và hiển thị trang chủ, chi tiết concert, đăng nhập, đăng ký, đơn vé. 3 mẫu mỗi trang trong tải steady. | Mọi mẫu thành công ≤ **3.000 ms**; lỗi, timeout và dữ liệu thiếu không được coi là đạt. |
| NFR02 | Đo riêng GET `/concerts`, `/concerts?on_sale=true`, `/concerts/{id}` trong steady. | Trung bình của từng API và tổng mẫu **< 2.000 ms**; có request thành công thực tế, không bỏ mẫu lỗi. |
| NFR03 | Tăng đến **100 user**, đợi tất cả đăng nhập/sẵn sàng rồi đo thêm **120 giây**, ghi số user sẵn sàng mỗi 0,5 giây. | Duy trì ≥100 user trong toàn bộ steady, đủ request, không lỗi khởi tạo/kịch bản. |
| NFR04 | Burst **100 người mua 20 vé**, mỗi request mua 1 vé; đọc tồn kho và đơn đã commit. | Đúng 20 đơn thành công, phần còn lại `409 SoldOut`; tồn kho không âm và `remaining + confirmed = total`. |
| NFR05 | Hủy các đơn khác nhau, 5 lượt hủy trùng cùng đơn, mua/hủy đồng thời; lấy snapshot tồn kho. | Không mất cập nhật/hoàn vé hai lần; mỗi đơn hủy trùng chỉ 1 response thành công, các lần còn lại `409 InvalidState`. |

**Giả định đo được công khai:** “mạng bình thường” được mô phỏng trong Chromium:
latency 50 ms, tải xuống 10 Mbps, tải lên 5 Mbps (có thể chỉnh ở ô 3).
NFR03 chưa quy định tỷ lệ lỗi; mặc định đặt ngưỡng lỗi bất thường **≤1%**, có thể chọn 0%.
120 giây và 3 mẫu/trang là quy ước kiểm thử, không phải số liệu đã đạt.
PASS chỉ áp dụng cho lượt đo và mô hình mạng này; không suy ra hiệu năng trên mọi mạng thật.

Tạo tài khoản/bcrypt nằm ngoài thời gian đo. Đăng nhập và ramp-up có log riêng;
bảng Locust tổng gồm cả warmup/steady/drain, bảng **NFR** dùng mẫu steady.
Workload giữ tối đa một đơn/user, một concert mở bán với 150 vé, nghỉ 0,5–2 giây/tác vụ.
Các bài tranh chấp chạy sau khi tải dừng, trên concert/hạng vé riêng trong database thử nghiệm.
Nếu SQLite nghẽn hoặc cancellation có race, báo cáo ghi **FAIL**, không sửa backend để che lỗi.

Kết quả gồm **PASS / FAIL / INCONCLUSIVE (chưa đủ bằng chứng)** cho từng NFR.
Chạy ít nhất **3 lượt** cùng cấu hình và giữ mọi ZIP để so sánh.
Nguồn API công cụ: [Locust](https://docs.locust.io/en/stable/api.html),
[Playwright CDP](https://playwright.dev/python/docs/api/class-cdpsession),
[Chromium network emulation](https://chromedevtools.github.io/devtools-protocol/tot/Network/#method-emulateNetworkConditions).

## Ô 1 — Giải nén mã nguồn đi kèm
Chạy ô này, không cần chỉnh chuỗi dữ liệu dài bên dưới.

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, tempfile, zipfile
from IPython.display import display, FileLink

# Kaggle uses Linux; a root-relative folder on Windows must not select Kaggle.
ON_KAGGLE = sys.platform.startswith("linux") and Path("/kaggle/working").is_dir()
WORKING = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "kaggle_runs").resolve()
WORKING.mkdir(parents=True, exist_ok=True)
print("Environment:", "Kaggle" if ON_KAGGLE else "Local", "| Output:", WORKING)
SOURCE_SHA256 = "c3afcdecfc0d5333fdfb6c54cbfdc44758c478acbf59d948294323396f61438d"
SOURCE_ARCHIVE = "..."
payload = base64.b64decode(SOURCE_ARCHIVE)
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256
REPO = Path(tempfile.mkdtemp(prefix="hit_the_vibe_source_", dir=WORKING))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        target = (REPO / name).resolve()
        assert target.is_relative_to(REPO.resolve()), "Đường dẫn không hợp lệ trong gói nguồn"
    archive.extractall(REPO)
print("Mã nguồn:", REPO)
print("SHA256:", SOURCE_SHA256)
print("Python:", sys.version, "| CPU logic:", os.cpu_count())

## Ô 2 — Cài thư viện và Chromium vào môi trường riêng
Giữ cách tạo venv không dùng ensurepip. Cài browser headless cùng thư viện hệ thống
trên Kaggle/Linux để đo **trang hiển thị thực tế**. Bước đầu có thể mất vài phút.
Chỉ tiếp tục khi ô báo thành công; nếu browser không cài được, không có bằng chứng NFR01.

In [ ]:
BASE_ENV = os.environ.copy()
for key in ("PYTHONPATH", "PYTHONHOME"):
    BASE_ENV.pop(key, None)
BASE_ENV.update(PYTHONNOUSERSITE="1", PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1")
VENV = Path(tempfile.mkdtemp(prefix="hit-the-vibe-env-", dir=WORKING))
PYTHON = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
SETUP_LOG = WORKING / f"{VENV.name}-setup.log"
def setup(command, **kwargs):
    with SETUP_LOG.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + repr(command) + "\n")
        log.flush()
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                **{key: value for key, value in kwargs.items() if key != "check"})
    if result.returncode:
        print(SETUP_LOG.read_text(encoding="utf-8", errors="replace")[-12000:])
        raise RuntimeError(f"Setup failed (exit {result.returncode}). Log: {SETUP_LOG}")
    print("OK:", " ".join(map(str, command[:4])))
print("Installing dependencies; please wait. Log:", SETUP_LOG)
setup([sys.executable, "-m", "venv", "--without-pip", str(VENV)],
               env=BASE_ENV, cwd=WORKING, check=True)
setup([sys.executable, "-m", "pip", "--python", str(PYTHON), "install", "--upgrade", "pip"],
               env=BASE_ENV, cwd=WORKING, check=True)
setup([str(PYTHON), "-m", "pip", "install", "wrapt>=1.16,<3",
                "-r", str(REPO / "loadtest" / "requirements.txt")],
               env=BASE_ENV, cwd=WORKING, check=True)
setup([str(PYTHON), "-m", "pip", "check"], env=BASE_ENV, check=True)
setup([str(PYTHON), "-m", "locust", "--version"], env=BASE_ENV, check=True)

BROWSER_DIR = WORKING / "nfr-chromium"
BASE_ENV["PLAYWRIGHT_BROWSERS_PATH"] = str(BROWSER_DIR)
browser_command = [str(PYTHON), "-m", "playwright", "install", "--only-shell", "chromium"]
if os.name != "nt":
    browser_command.insert(-1, "--with-deps")
setup(browser_command, env=BASE_ENV, cwd=WORKING, check=True)
print("Thư viện và Chromium đã cài xong. Tiếp tục ô 3.")

## Ô 3 — Cấu hình một lượt NFR mới
Mặc định: **100 user, 120 giây steady sau warmup**. Có thể tăng user lên 200.
Dưới 100 user hoặc steady dưới 120 giây chỉ là smoke test: NFR03 không được PASS.
Khi chạy thêm lượt, chạy lại **ô 3 → 5**, không chạy đồng thời hai lượt.

In [ ]:
from pathlib import Path
import sys, tempfile

# Recompute the output directory for every run, including an already-open kernel.
ON_KAGGLE = sys.platform.startswith("linux") and Path("/kaggle/working").is_dir()
WORKING = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "kaggle_runs").resolve()
WORKING.mkdir(parents=True, exist_ok=True)

PHASE = "1"
USERS = 100
SPAWN_RATE = 20
STEADY_SECONDS = 120
WARMUP_TIMEOUT_SECONDS = 180
TICKETS = 150
PAGE_SAMPLES_PER_ROUTE = 3
NORMAL_NETWORK_LATENCY_MS = 50
NORMAL_NETWORK_DOWNLOAD_MBPS = 10
NORMAL_NETWORK_UPLOAD_MBPS = 5
MAX_UNEXPECTED_ERROR_RATE = 0.01  # Giả định cho NFR03; đổi thành 0 nếu yêu cầu không lỗi.
RANDOM_SEED = 1211
PORT = 8000
RUN_DIR = Path(tempfile.mkdtemp(prefix=f"phase{PHASE}_nfr_", dir=WORKING))
print("Kết quả lượt mới:", RUN_DIR)
print(f"{USERS} user; steady {STEADY_SECONDS}s; warmup timeout {WARMUP_TIMEOUT_SECONDS}s")

## Ô 4 — Chạy tải, đo trang và kiểm tra tranh chấp vé
Khởi tạo tài khoản trước khi đo; giữ timeout khởi tạo và lỗi request đúng thực tế.
Đợi đủ user sẵn sàng → đo steady và trang → drain request → chạy bài tranh chấp → dừng API.
Không đạt đủ user sẽ không giả lập một giai đoạn steady thành công.

Mã thoát **0**: cả 5 NFR PASS; **1**: có NFR FAIL; **2**: lỗi runner hoặc INCONCLUSIVE.
Cả 1 và 2 vẫn xuất báo cáo/log/ZIP khi runner đã bắt đầu. Nếu ngắt ô, đợi dọn dẹp xong
và đọc ô 5. Không bấm Restart Session trước khi tải ZIP.

In [ ]:
command = [str(PYTHON), "-m", "loadtest.run", "--output-dir", str(RUN_DIR),
           "--users", str(USERS), "--spawn-rate", str(SPAWN_RATE),
           "--seconds", str(STEADY_SECONDS), "--warmup-seconds", str(WARMUP_TIMEOUT_SECONDS),
           "--tickets", str(TICKETS), "--page-samples", str(PAGE_SAMPLES_PER_ROUTE),
           "--latency-ms", str(NORMAL_NETWORK_LATENCY_MS),
           "--download-mbps", str(NORMAL_NETWORK_DOWNLOAD_MBPS),
           "--upload-mbps", str(NORMAL_NETWORK_UPLOAD_MBPS),
           "--max-error-rate", str(MAX_UNEXPECTED_ERROR_RATE),
           "--seed", str(RANDOM_SEED), "--phase", PHASE, "--port", str(PORT)]
import signal
launch_flags = {"creationflags": subprocess.CREATE_NO_WINDOW} if os.name == "nt" else {}
benchmark = subprocess.Popen(command, cwd=REPO, env=BASE_ENV, **launch_flags)
try:
    return_code = benchmark.wait()
except KeyboardInterrupt:
    print("Đang yêu cầu dừng và lưu kết quả; vui lòng chờ...")
    if os.name == "nt":
        (RUN_DIR / "stop-run").touch()
    else:
        benchmark.send_signal(signal.SIGINT)
    try:
        return_code = benchmark.wait(timeout=55)
    except subprocess.TimeoutExpired:
        benchmark.kill()
        benchmark.wait()
        raise RuntimeError("Runner không dừng kịp. Kiểm tra tiến trình/log trước khi chạy tiếp.")
print("Mã thoát:", return_code, "| Kết quả:", RUN_DIR)
if return_code:
    print("Đọc bảng từng NFR và log ở ô 5; không kết luận đạt từ cấu hình số user.")

## Ô 5 — Đọc và tải kết quả
- **nfr_summary.txt**, **nfr_results.csv**, **nfr_results.json**: kết luận từng NFR, ngưỡng, số liệu và lý do.
- **pages.json**, **pages.log**: thời gian hiển thị 5 trang và mô hình mạng; ảnh khi lỗi render.
- **requests.csv**, **concurrency.csv**, **scenario.json**: từng request theo phase và số user sẵn sàng theo thời gian.
- **concurrency.json**, **concurrency.log**: bài tranh chấp độc lập, response và invariant tồn kho.
- **baseline_summary.txt**, **final_stats.csv**, **report.html**, **inventory_check.csv**: workload săn vé gốc (không gồm fixture tranh chấp).
- **resources.csv**, **api.log**, **locust.log**, **metadata.json**, **requirements-resolved.txt**, **source/**: môi trường, tài nguyên, lỗi và mã đã chạy.

`409 SoldOut` là từ chối hết vé hợp lệ, không phải đơn thành công. Hủy trùng chấp nhận
`409 InvalidState` **chỉ trong bài kiểm tra hủy trùng**. HTTP 500, timeout, phản hồi sai
hoặc không có đơn thành công không được tự chuyển thành PASS.

In [ ]:
summary = RUN_DIR / "nfr_summary.txt"
if summary.exists():
    print(summary.read_text(encoding="utf-8"))
    report = json.loads((RUN_DIR / "nfr_results.json").read_text(encoding="utf-8"))
    for row in report["results"]:
        print("\n", row["nfr"], row["status"])
        print("Số liệu:", json.dumps(row["observed"], ensure_ascii=False, indent=2))
else:
    print("Chưa có báo cáo hoàn chỉnh; kiểm tra log.")
for name in ("nfr_report_error.txt", "report_error.txt", "prepare_tail.txt", "api_tail.txt", "locust_tail.txt"):
    path = RUN_DIR / name
    if path.exists():
        display(FileLink(str(path)))
for path in (Path(str(RUN_DIR) + ".zip"), summary, RUN_DIR / "nfr_results.csv",
             RUN_DIR / "nfr_results.json", RUN_DIR / "report.html"):
    if path.exists():
        display(FileLink(str(path)))
print("Tải ZIP để giữ mọi bằng chứng và log khi phiên Kaggle kết thúc.")

## Khi cần xem lỗi API trực tiếp
Chạy ô dưới sau khi ô 4 kết thúc. Không cần nhớ tên thư mục của lượt trước.

In [ ]:
log_path = RUN_DIR / "api.log"
if log_path.exists():
    print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-200:]))
else:
    print("API chưa khởi động. Kiểm tra prepare.log hoặc thông báo ở ô 4.")
    prepare_log = RUN_DIR / "prepare.log"
    if prepare_log.exists():
        print(prepare_log.read_text(encoding="utf-8", errors="replace")[-6000:])